In [ ]:
# # Multi-Layer Perceptron (MLP) Classifier
#
# This notebook explores the training, evaluation, and performance analysis of a **Multi-Layer Perceptron (MLP)** Feedforward Neural Network using Scikit-Learn.
#
# ### Mathematical Concepts & Intuition
# A Multi-Layer Perceptron (MLP) is a classic feedforward neural network structure consisting of an input layer, one or more hidden layers, and an output layer.
# Neurons inside hidden layers compute a weighted sum of their inputs, add a bias, and apply a non-linear activation function:
# $$a^{(l)} = f(W^{(l)} a^{(l-1)} + b^{(l)})$$
#
# For backpropagation, the network computes gradients of the loss function with respect to weights and updates them iteratively to minimize prediction error.
#
# ### Hyperparameters Tuned
# 1. **Hidden Layer Configurations (`hidden_layer_sizes`)**:
#    - `(100,)`: 1 hidden layer containing 100 neurons.
#    - `(100, 50)`: 2 hidden layers with 100 neurons first, then 50 neurons.
#    - `(128, 64, 32)`: 3 hidden layers for deep representation capacity.
# 2. **Activation Functions**:
#    - `relu`: Rectified Linear Unit ($f(x) = \max(0, x)$). Prevents gradient saturation.
#    - `tanh`: Hyperbolic Tangent ($f(x) = \tanh(x)$). Outputs zero-centered data values.
# 3. **Max Iterations**: Constrained to `200` to allow sufficient epochs for weight optimization.


In [ ]:
# Step 1: Imports and setup
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)


In [ ]:
# ### Step 2: Data Loading
# Load processed training datasets.


In [ ]:
# Step 2: Load the preprocessed dataset
PROCESSED_DATA_PATH = "../data/processed_data.joblib"
if not os.path.exists(PROCESSED_DATA_PATH):
    raise FileNotFoundError(f"Preprocessed data not found at {PROCESSED_DATA_PATH}")

data = joblib.load(PROCESSED_DATA_PATH)
X_train, X_test, y_train, y_test = data['X_train'], data['X_test'], data['y_train'], data['y_test']
print("Preprocessed data values imported.")


In [ ]:
# ### Step 3: High-Iteration Training Grid Search
# We fit MLPs across the configuration grid and record loss behaviors. We evaluate different configurations of **hidden layers** to see how network depth/width affects accuracy.


In [ ]:
# Step 3: Grid Search Fitting Loop
# Explanation of Hidden Layer Configurations:
# - (100,): 1 hidden layer with 100 neurons (shallow network, fits linear/moderate relationships).
# - (100, 50): 2 hidden layers (first has 100 neurons, second has 50). Learns structured combinations of weather factors.
# - (128, 64, 32): 3 hidden layers (deep network, gradually condenses 128 features to 64, then 32, extracting highly complex non-linear abstractions).
hidden_configs = [(100,), (100, 50), (128, 64, 32)]
activations = ['relu', 'tanh']

results = []
best_acc = 0
best_model = None

print("--- Starting MLP Classifier Tuning Loop ---")
for layers in hidden_configs:
    for activation in activations:
        print(f"Training MLP: hidden_layers={str(layers):<15} | activation={activation:<6}")
        # Setup model
        model = MLPClassifier(hidden_layer_sizes=layers, activation=activation, 
                              max_iter=200, random_state=42)
        model.fit(X_train, y_train)
        
        # Evaluate accuracy
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        print(f"  --> Accuracy: {acc * 100:.2f}%")
        
        results.append({
            'hidden_layers': layers,
            'activation': activation,
            'accuracy': acc
        })
        
        if acc > best_acc:
            best_acc = acc
            best_model = model

print("\nGrid training complete!")


In [ ]:
# ### Step 4: Metric Evaluation & Learning Curve Visualizations
# We print scores and plot both the Confusion Matrix Heatmap and the Multi-Iteration **Loss Optimization Curve**.


In [ ]:
# Step 4: Evaluation and Plotting
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=== BEST MODEL METRICS ===")
print(f"Optimal Layers Size: {best_model.hidden_layer_sizes}")
print(f"Optimal Activation:  {best_model.activation}")
print(f"Test Accuracy:       {accuracy * 100:.2f}%")
print(f"Precision Score:     {precision * 100:.2f}%")
print(f"Recall Score:        {recall * 100:.2f}%")
print(f"F1 Performance:      {f1 * 100:.2f}%")

# Set up multi-figure layouts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
            xticklabels=['No Rain', 'Rain'], yticklabels=['No Rain', 'Rain'], ax=axes[0])
axes[0].set_title('Confusion Matrix - MLP (Best Model)', fontsize=13, pad=10)
axes[0].set_ylabel('Actual Label')
axes[0].set_xlabel('Predicted Label')

# 2. Plot MLP Loss Minimization Curve
axes[1].plot(best_model.loss_curve_, color='forestgreen', linewidth=2)
axes[1].set_title('Backpropagation Loss Curve (Best Model)', fontsize=13, pad=10)
axes[1].set_xlabel('Iterations (Epochs)')
axes[1].set_ylabel('Loss Metric (Cross-Entropy)')
axes[1].grid(True, linestyle='--')

plt.tight_layout()
plt.show()


In [ ]:
# ### Step 5: Save Trained MLP
# Save weights to `../data/models/mlp.joblib`.


In [ ]:
# Step 5: Export best MLP
MODELS_DIR = "../data/models"
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)

model_path = os.path.join(MODELS_DIR, 'mlp.joblib')
joblib.dump(best_model, model_path)
print(f"Optimal MLP model successfully exported to: {model_path}")
